In [57]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [58]:
train_path = Path(r"..\..\data\clean\cleaned_data.csv")
df = pd.read_csv(train_path)

In [59]:
numberical_continuos=["LotArea", "GrLivArea", "TotalBsmtSF", "GarageArea", "MasVnrArea", "LotFrontage"]
numberical_count=["FullBath", "HalfBath", "BedroomAbvGr", "TotRmsAbvGrd", "GarageCars", "Fireplaces"]
ordinal_categorical=["ExterQual", "ExterCond", "KitchenQual", "HeatingQC", "BsmtQual", "BsmtCond", "GarageQual", "GarageCond", "FireplaceQu"]
nominal_categorical= ["Neighborhood", "MSZoning", "BldgType", "HouseStyle", "RoofStyle", "Exterior1st", "Exterior2nd", "SaleType", "SaleCondition"]
Binary= ["CentralAir", "Street"]
Special_Coded_Category= ["MSSubClass", "MoSold", "YrSold"]
Target= ["SalePrice"]
Id=["Id"]

# xử lý riêng các feature đặc biệt

* MSSubClass: One-Hot Encoding vì đây là mã loại nhà, không phải số đo.
* MoSold: One-Hot Encoding vì tháng 12 không có nghĩa là lớn hơn tháng 1 về mặt định lượng.
* YrSold: StandardScaler vì có thể dùng như biến thời gian dạng số.

In [60]:
quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]

ordinal_orders = []

for col in ordinal_categorical:
    if col in ["BsmtQual", "BsmtCond"]:
        order = ["No Basement"] + quality_order

    elif col in ["GarageQual", "GarageCond"]:
        order = ["No Garage"] + quality_order

    elif col == "FireplaceQu":
        order = ["No Fireplace"] + quality_order

    else:
        order = quality_order

    ordinal_orders.append(order)

# xây dựng pipeline xử lý

In [61]:
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)
from sklearn.impute import SimpleImputer

# Numerical
numeric_cols = (
    numberical_continuos
    + numberical_count
    + ["YrSold"]
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Ordinal
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_orders,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# Nominal
nominal_cols = (
    nominal_categorical
    + ["MSSubClass", "MoSold"]
)

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Binary
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="if_binary",
        handle_unknown="ignore"
    ))
])

# Combine
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("ord", ordinal_pipeline, ordinal_categorical),
        ("nom", nominal_pipeline, nominal_cols),
        ("bin", binary_pipeline, Binary)
    ],
    remainder="drop"
)



# chia train và valid

In [62]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["SalePrice", "Id"])
y = df["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,y,test_size=0.2,random_state=42
)


# fit và transforms trên train

In [63]:
X_train_processed = preprocessor.fit_transform(X_train)

# chỉ trasnfrom trên valid

In [64]:
X_valid_processed = preprocessor.transform(X_valid)
feature_names = preprocessor.get_feature_names_out()

In [65]:
import pandas as pd
from scipy import sparse

if sparse.issparse(X_train_processed):
    X_train_processed = X_train_processed.toarray()
    X_valid_processed = X_valid_processed.toarray()

X_train_final = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_valid_final = pd.DataFrame(
    X_valid_processed,
    columns=feature_names,
    index=X_valid.index
)

In [66]:
from pathlib import Path
import joblib

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

X_train_final.to_csv(output_dir / "X_train.csv", index=False)
X_valid_final.to_csv(output_dir / "X_test.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_valid.to_csv(output_dir / "y_test.csv", index=False)

joblib.dump(
    preprocessor,
    output_dir / "preprocessor.joblib"
)

['data\\processed\\preprocessor.joblib']

In [67]:
test_path=Path(r"..\..\data\clean\cleaned_test.csv")

In [68]:
test_df=pd.read_csv(test_path)

In [69]:
test_df=test_df.drop(columns='Id')

In [70]:
X_test_processed = preprocessor.transform(test_df)
test_feature_names = preprocessor.get_feature_names_out()
import pandas as pd
from scipy import sparse

if sparse.issparse(X_test_processed):
    X_test_processed = X_test_processed.toarray()

X_test_final = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out(),
    index=test_df.index
)
X_test_final.to_csv(
    "data/processed/X_kaggle_test.csv",
    index=False
)